# Analisi Statistiche Merge
Analisi dettagliata delle statistiche dei file di merge per combinazioni di FSVERSION, METHOD_CSF e METHOD_PET

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Configurazione grafici
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Carica i dati
df = pd.read_csv('merge_statistics.csv')
print(f"File caricati: {len(df)}")
df.head()

## 1. Overview delle Combinazioni

In [ ]:
# Combinazioni uniche
print("=" * 60)
print("COMBINAZIONI UNICHE")
print("=" * 60)
print(f"\nFSVERSION disponibili: {df['FSVERSION'].unique().tolist()}")
print(f"METHOD_CSF disponibili: {df['METHOD_CSF'].unique().tolist()}")
print(f"METHOD_PET disponibili: {df['METHOD_PET'].unique().tolist()}")
print(f"\nTotale combinazioni: {len(df)}")

# Tabella pivot delle combinazioni
print("\n" + "=" * 60)
print("MATRICE COMBINAZIONI (righe per file)")
print("=" * 60)
pivot = df.pivot_table(index=['FSVERSION'], 
                       columns=['METHOD_CSF', 'METHOD_PET'], 
                       values='n_righe', 
                       aggfunc='first')
display(pivot)

## 2. Statistiche Descrittive Generali

In [ ]:
# Statistiche numeriche principali
cols_numeriche = ['n_righe', 'n_colonne', 'n_soggetti', 'visite_media', 'visite_max', 'nan_medio_pct', 'nan_max_pct']
print("STATISTICHE DESCRITTIVE PRINCIPALI")
print("=" * 60)
display(df[cols_numeriche].describe().round(2))

In [ ]:
# Tabella riassuntiva formattata
summary = df[['FSVERSION', 'METHOD_CSF', 'METHOD_PET', 'n_righe', 'n_soggetti', 
              'visite_media', 'nan_medio_pct', 'nan_max_pct']].copy()
summary['combinazione'] = summary['FSVERSION'].astype(str) + ' | ' + summary['METHOD_CSF'] + ' | ' + summary['METHOD_PET']
summary = summary.sort_values(['FSVERSION', 'METHOD_CSF', 'METHOD_PET'])

print("\nTABELLA RIASSUNTIVA PER COMBINAZIONE")
print("=" * 80)
display(summary[['combinazione', 'n_righe', 'n_soggetti', 'visite_media', 'nan_medio_pct', 'nan_max_pct']].set_index('combinazione'))

## 3. Analisi NaN

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Grafico 1: NaN medio per combinazione
df_sorted = df.sort_values('nan_medio_pct')
colors = ['#e74c3c' if x == 'lumipulse' else '#3498db' for x in df_sorted['METHOD_CSF']]
bars = axes[0].barh(range(len(df_sorted)), df_sorted['nan_medio_pct'], color=colors)
axes[0].set_yticks(range(len(df_sorted)))
axes[0].set_yticklabels([f"{r['FSVERSION']}|{r['METHOD_CSF'][:3]}|{r['METHOD_PET']}" 
                         for _, r in df_sorted.iterrows()], fontsize=8)
axes[0].set_xlabel('NaN Medio (%)')
axes[0].set_title('NaN Medio % per Combinazione')
axes[0].axvline(df['nan_medio_pct'].mean(), color='black', linestyle='--', label=f"Media: {df['nan_medio_pct'].mean():.1f}%")
axes[0].legend()

# Grafico 2: Boxplot NaN per METHOD_CSF
df.boxplot(column='nan_medio_pct', by='METHOD_CSF', ax=axes[1])
axes[1].set_title('Distribuzione NaN Medio per METHOD_CSF')
axes[1].set_xlabel('METHOD_CSF')
axes[1].set_ylabel('NaN Medio (%)')
plt.suptitle('')

plt.tight_layout()
plt.show()

# Statistiche NaN
print("\nSTATISTICHE NaN PER METHOD_CSF")
print(df.groupby('METHOD_CSF')[['nan_medio_pct', 'nan_max_pct']].agg(['mean', 'min', 'max']).round(2))

In [ ]:
# Heatmap NaN medio per FSVERSION e metodi
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Per METHOD_CSF
pivot_csf = df.pivot_table(index='FSVERSION', columns='METHOD_CSF', values='nan_medio_pct', aggfunc='mean')
sns.heatmap(pivot_csf, annot=True, fmt='.2f', cmap='RdYlGn_r', ax=axes[0])
axes[0].set_title('NaN Medio % per FSVERSION e METHOD_CSF')

# Per METHOD_PET
pivot_pet = df.pivot_table(index='FSVERSION', columns='METHOD_PET', values='nan_medio_pct', aggfunc='mean')
sns.heatmap(pivot_pet, annot=True, fmt='.2f', cmap='RdYlGn_r', ax=axes[1])
axes[1].set_title('NaN Medio % per FSVERSION e METHOD_PET')

plt.tight_layout()
plt.show()

## 4. Analisi Righe e Visite

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Grafico 1: N. righe per FSVERSION
df.groupby('FSVERSION')['n_righe'].mean().plot(kind='bar', ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('Media Righe per FSVERSION')
axes[0, 0].set_xlabel('FSVERSION')
axes[0, 0].set_ylabel('N. Righe')
axes[0, 0].tick_params(axis='x', rotation=45)

# Grafico 2: N. righe per combinazione METHOD
df['method_combo'] = df['METHOD_CSF'] + ' + ' + df['METHOD_PET']
df.groupby('method_combo')['n_righe'].mean().plot(kind='bar', ax=axes[0, 1], color='coral')
axes[0, 1].set_title('Media Righe per Combinazione Metodi')
axes[0, 1].set_xlabel('Combinazione')
axes[0, 1].set_ylabel('N. Righe')
axes[0, 1].tick_params(axis='x', rotation=45)

# Grafico 3: Visite media per FSVERSION
df.groupby('FSVERSION')['visite_media'].mean().plot(kind='bar', ax=axes[1, 0], color='green')
axes[1, 0].set_title('Media Visite per FSVERSION')
axes[1, 0].set_xlabel('FSVERSION')
axes[1, 0].set_ylabel('Visite Media')
axes[1, 0].tick_params(axis='x', rotation=45)

# Grafico 4: Distribuzione visite_max
df.boxplot(column='visite_max', by='FSVERSION', ax=axes[1, 1])
axes[1, 1].set_title('Distribuzione Visite Max per FSVERSION')
axes[1, 1].set_xlabel('FSVERSION')
axes[1, 1].set_ylabel('Visite Max')
plt.suptitle('')

plt.tight_layout()
plt.show()

## 5. Confronto Dettagliato per Versione

In [ ]:
# Tabella comparativa per FSVERSION
print("CONFRONTO PER FSVERSION")
print("=" * 80)
comparison = df.groupby('FSVERSION').agg({
    'n_righe': ['mean', 'std'],
    'visite_media': 'mean',
    'visite_max': 'max',
    'nan_medio_pct': ['mean', 'min', 'max'],
    'colonne_con_nan': 'mean'
}).round(2)
comparison.columns = ['righe_media', 'righe_std', 'visite_media', 'visite_max', 
                      'nan_medio', 'nan_min', 'nan_max', 'col_con_nan']
display(comparison)

In [ ]:
# Confronto per METHOD_CSF
print("\nCONFRONTO PER METHOD_CSF")
print("=" * 80)
comparison_csf = df.groupby('METHOD_CSF').agg({
    'n_righe': ['mean', 'std'],
    'visite_media': 'mean',
    'nan_medio_pct': ['mean', 'min', 'max'],
}).round(2)
display(comparison_csf)

# Confronto per METHOD_PET
print("\nCONFRONTO PER METHOD_PET")
print("=" * 80)
comparison_pet = df.groupby('METHOD_PET').agg({
    'n_righe': ['mean', 'std'],
    'visite_media': 'mean',
    'nan_medio_pct': ['mean', 'min', 'max'],
}).round(2)
display(comparison_pet)

## 6. Heatmap Completa

In [ ]:
# Heatmap con tutte le combinazioni
df['combo'] = df['METHOD_CSF'] + ' | ' + df['METHOD_PET']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Heatmap n_righe
pivot_righe = df.pivot_table(index='FSVERSION', columns='combo', values='n_righe')
sns.heatmap(pivot_righe, annot=True, fmt='.0f', cmap='Blues', ax=axes[0])
axes[0].set_title('N. Righe per Combinazione')

# Heatmap visite_media
pivot_visite = df.pivot_table(index='FSVERSION', columns='combo', values='visite_media')
sns.heatmap(pivot_visite, annot=True, fmt='.2f', cmap='Greens', ax=axes[1])
axes[1].set_title('Visite Media per Combinazione')

# Heatmap nan_medio_pct
pivot_nan = df.pivot_table(index='FSVERSION', columns='combo', values='nan_medio_pct')
sns.heatmap(pivot_nan, annot=True, fmt='.2f', cmap='RdYlGn_r', ax=axes[2])
axes[2].set_title('NaN Medio % per Combinazione')

plt.tight_layout()
plt.show()

## 7. Ranking e Best/Worst Cases

In [ ]:
# File con meno NaN
print("TOP 5 FILE CON MENO NaN (MIGLIORI)")
print("=" * 80)
best = df.nsmallest(5, 'nan_medio_pct')[['FSVERSION', 'METHOD_CSF', 'METHOD_PET', 'n_righe', 'nan_medio_pct', 'nan_max_pct']]
display(best)

print("\nTOP 5 FILE CON PIU' NaN (PEGGIORI)")
print("=" * 80)
worst = df.nlargest(5, 'nan_medio_pct')[['FSVERSION', 'METHOD_CSF', 'METHOD_PET', 'n_righe', 'nan_medio_pct', 'nan_max_pct']]
display(worst)

In [ ]:
# File con piu' righe (piu' dati)
print("TOP 5 FILE CON PIU' RIGHE")
print("=" * 80)
most_rows = df.nlargest(5, 'n_righe')[['FSVERSION', 'METHOD_CSF', 'METHOD_PET', 'n_righe', 'visite_media', 'nan_medio_pct']]
display(most_rows)

## 8. Correlazioni

In [ ]:
# Matrice di correlazione
cols_corr = ['n_righe', 'visite_media', 'visite_max', 'visite_std', 'nan_medio_pct', 'nan_max_pct']
corr_matrix = df[cols_corr].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            square=True, linewidths=0.5)
plt.title('Matrice di Correlazione')
plt.tight_layout()
plt.show()

print("\nCORRELAZIONI SIGNIFICATIVE:")
for i in range(len(cols_corr)):
    for j in range(i+1, len(cols_corr)):
        corr = corr_matrix.iloc[i, j]
        if abs(corr) > 0.5:
            print(f"  {cols_corr[i]} <-> {cols_corr[j]}: {corr:.3f}")

## 9. Riepilogo Finale

In [ ]:
print("=" * 80)
print("RIEPILOGO FINALE")
print("=" * 80)

print(f"\n{'Totale file analizzati:':<40} {len(df)}")
print(f"{'Versioni FreeSurfer:':<40} {len(df['FSVERSION'].unique())} ({', '.join(map(str, df['FSVERSION'].unique()))})")
print(f"{'Metodi CSF:':<40} {len(df['METHOD_CSF'].unique())} ({', '.join(df['METHOD_CSF'].unique())})")
print(f"{'Metodi PET:':<40} {len(df['METHOD_PET'].unique())} ({', '.join(df['METHOD_PET'].unique())})")

print(f"\n{'--- Statistiche Righe ---':<40}")
print(f"{'Media righe per file:':<40} {df['n_righe'].mean():.0f}")
print(f"{'Min righe:':<40} {df['n_righe'].min():.0f}")
print(f"{'Max righe:':<40} {df['n_righe'].max():.0f}")

print(f"\n{'--- Statistiche Visite ---':<40}")
print(f"{'Media visite (media dei file):':<40} {df['visite_media'].mean():.2f}")
print(f"{'Max visite osservate:':<40} {df['visite_max'].max():.0f}")

print(f"\n{'--- Statistiche NaN ---':<40}")
print(f"{'NaN medio % (media file):':<40} {df['nan_medio_pct'].mean():.2f}%")
print(f"{'NaN medio % minimo:':<40} {df['nan_medio_pct'].min():.2f}%")
print(f"{'NaN medio % massimo:':<40} {df['nan_medio_pct'].max():.2f}%")

print(f"\n{'--- Migliore Combinazione (meno NaN) ---':<40}")
best_row = df.loc[df['nan_medio_pct'].idxmin()]
print(f"FSVERSION: {best_row['FSVERSION']}, METHOD_CSF: {best_row['METHOD_CSF']}, METHOD_PET: {best_row['METHOD_PET']}")
print(f"NaN medio: {best_row['nan_medio_pct']:.2f}%, Righe: {best_row['n_righe']:.0f}")

print(f"\n{'--- Peggiore Combinazione (piu NaN) ---':<40}")
worst_row = df.loc[df['nan_medio_pct'].idxmax()]
print(f"FSVERSION: {worst_row['FSVERSION']}, METHOD_CSF: {worst_row['METHOD_CSF']}, METHOD_PET: {worst_row['METHOD_PET']}")
print(f"NaN medio: {worst_row['nan_medio_pct']:.2f}%, Righe: {worst_row['n_righe']:.0f}")